# Covariate-adjusted residual heterogeneity

The paper uses a fixed trigonometric feature kernel for nuisance regression, with CV ridge penalties, and a fixed 16-feature Gaussian RFF approximation for the residual kernel. All methods share the same data and features.

Delta confidence sets are centered at U on the squared-norm scale, intersected with the nonnegative axis, then square-root transformed. All repetitions enter coverage and mean-width summaries; empty sets have noncoverage and zero width. See `CAUSAL_COVARIATE_REPRODUCTION.md`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from statistics import NormalDist
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "semibold",
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 9.5,
        "axes.labelsize": 9.5,
        "axes.titlesize": 10.0,
        "legend.fontsize": 8.5,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
    }
)

N = 120 # 120 is good
N_FOLDS = 4
N_REPS = 120

# Honest basis size: p = 2 + 4 * MAX_FREQ = 22 parameters.
# Each outer fold trains on 90 points, so finite-sample error is real but not manufactured by misspecification.
MAX_FREQ = 5
MEAN_AMPLITUDE = 1.5

FIGURE_DIR = Path("figures/toy_residual_hsic_with_covariate")
SAVE_FIGURES = True
SAVE_FORMATS = ("png", "pdf")

# Edit these values to change the homoskedastic null and X-scale alternative.
# The named "null" is genuinely null when the two X-specific sigmas are equal.
REGIME_SIGMA_BY_X = {
    "null": {0: 0.45, 1: 0.45},
    "x_scale": {0: 0.35, 1: 0.55},
}

# Fixed random Fourier features for a Gaussian residual kernel.
# The bandwidth should be comparable to the residual standard deviations in the DGP.
RFF_DIM = 16
RFF_LENGTH_SCALE = 1.0
RFF_SEED = 12_345

_rff_rng = np.random.default_rng(RFF_SEED)
RFF_OMEGA = _rff_rng.normal(loc=0.0, scale=1.0 / RFF_LENGTH_SCALE, size=RFF_DIM)
RFF_PHASE = _rff_rng.uniform(0.0, 2.0 * np.pi, size=RFF_DIM)
RFF_SCALE = np.sqrt(2.0 / RFF_DIM)

# False regenerates seeded repetitions; True redraws from the saved CSVs.
LOAD_SAVED_RESULTS = False
RESULT_DIR = Path("results/covariate")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Register the paper font explicitly: an old matplotlib cache can omit system fonts.
from matplotlib import font_manager
for font_path in font_manager.findSystemFonts():
    if Path(font_path).name.startswith("LiberationSans"):
        font_manager.fontManager.addfont(font_path)
font_manager.findfont("Liberation Sans", fallback_to_default=False)
plt.rcParams["font.family"] = "Liberation Sans"


In [ ]:
def one_hot_binary(x):
    x = np.asarray(x).astype(int)
    return np.column_stack([1 - x, x]).astype(float)


def trig_design(x, z, max_freq):
    """Regression class used both for the truth and for the mean fit."""
    x = np.asarray(x).astype(float)
    z = np.asarray(z).astype(float)
    cols = [np.ones_like(z), x]
    for k in range(1, max_freq + 1):
        s = np.sin(k * z)
        c = np.cos(k * z)
        cols += [s, c, x * s, x * c]
    return np.column_stack(cols)


def ridge_fit(Phi, y, alpha):
    penalty = alpha * np.eye(Phi.shape[1])
    penalty[0, 0] = 0.0
    return np.linalg.solve(Phi.T @ Phi + penalty, Phi.T @ y)


def cv_ridge_fit(Phi, y, alphas=(1e-8, 1e-6, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0), n_splits=3):
    """Small inner-loop CV for a realistic nuisance fit."""
    idx = np.arange(len(y))
    folds = np.array_split(idx, n_splits)

    best_alpha = None
    best_score = np.inf
    for alpha in alphas:
        scores = []
        for val in folds:
            train = np.setdiff1d(idx, val)
            beta = ridge_fit(Phi[train], y[train], alpha)
            pred = Phi[val] @ beta
            scores.append(np.mean((pred - y[val]) ** 2))
        score = float(np.mean(scores))
        if score < best_score - 1e-12:
            best_score = score
            best_alpha = alpha

    return ridge_fit(Phi, y, best_alpha), float(best_alpha), best_score


def make_folds(n, n_folds, rng):
    return np.array_split(rng.permutation(n), n_folds)


def residual_feature_map(r):
    """Fixed random Fourier features for a Gaussian residual kernel."""
    r = np.asarray(r)
    return RFF_SCALE * np.cos(r[:, None] * RFF_OMEGA[None, :] + RFF_PHASE[None, :])


def residual_score_map(r):
    """Derivative phi'(r), used as the training target for the fitted v nuisance."""
    r = np.asarray(r)
    angle = r[:, None] * RFF_OMEGA[None, :] + RFF_PHASE[None, :]
    return -RFF_SCALE * RFF_OMEGA[None, :] * np.sin(angle)


def offdiag_average(M):
    """NumPy version of multivariate_crossfit_core.offdiag_average."""
    n = M.shape[0]
    return (np.sum(M) - np.trace(M)) / (n * (n - 1))


def crossfit_u_from_blocks(H_folds):
    """NumPy version of multivariate_crossfit_core.crossfit_u_from_blocks."""
    num_folds = len(H_folds)
    diag_sum = sum(offdiag_average(H_folds[i][i]) for i in range(num_folds))
    offdiag_sum = sum(
        np.mean(H_folds[i][j])
        for i in range(num_folds)
        for j in range(num_folds)
        if i != j
    )
    return float((diag_sum + offdiag_sum) / (num_folds ** 2))


def score_gram_blocks(A, B, folds):
    """Fold blocks of <A_i otimes B_i, A_j otimes B_j>."""
    rows = []
    for fold_i in folds:
        row = []
        for fold_j in folds:
            row.append((A[fold_i] @ A[fold_j].T) * (B[fold_i] @ B[fold_j].T))
        rows.append(row)
    return rows


def psi_from_contributions(A, B):
    psi = np.einsum("ia,ib->ab", A, B) / len(A)
    return float(np.linalg.norm(psi)), psi


def sigma_values_for_regime(regime):
    if regime not in REGIME_SIGMA_BY_X:
        raise ValueError(f"unknown regime: {regime}")
    values = REGIME_SIGMA_BY_X[regime]
    return float(values[0]), float(values[1])


def sigma_for_x(regime, x):
    sigma0, sigma1 = sigma_values_for_regime(regime)
    return np.where(np.asarray(x).astype(int) == 0, sigma0, sigma1)


def sigma_label_for_regime(regime):
    sigma0, sigma1 = sigma_values_for_regime(regime)
    if np.isclose(sigma0, sigma1):
        return rf"$\sigma(x)={sigma0:g}$"
    return rf"$\sigma(0)={sigma0:g},\;\sigma(1)={sigma1:g}$"


def population_oracle_hsic(regime):
    """True population HSIC = ||Cov(phi_X(X), phi_R(R))||^2 for the fixed RFF map."""
    sigma0, sigma1 = sigma_values_for_regime(regime)
    mu0 = RFF_SCALE * np.cos(RFF_PHASE) * np.exp(-0.5 * (sigma0 * RFF_OMEGA) ** 2)
    mu1 = RFF_SCALE * np.cos(RFF_PHASE) * np.exp(-0.5 * (sigma1 * RFF_OMEGA) ** 2)
    delta = mu0 - mu1
    psi = np.vstack([0.25 * delta, -0.25 * delta])
    return float(np.sum(psi ** 2))


In [ ]:
def true_mean(x, z):
    """The true mean is exactly in the span of trig_design(..., MAX_FREQ)."""
    x = np.asarray(x).astype(float)
    z = np.asarray(z).astype(float)
    mean = np.zeros_like(z)
    for k in range(1, MAX_FREQ + 1):
        mean += (0.30 / k) * np.sin(k * z) + (0.25 / k) * np.cos(k * z)
        mean += (0.22 / k) * x * np.sin(k * z) + (0.18 / k) * x * np.cos(k * z)
    mean += 0.2 * x
    return MEAN_AMPLITUDE * mean


def simulate(n=N, regime="null", seed=0):
    rng = np.random.default_rng(seed)
    x = rng.binomial(1, 0.5, size=n)
    z = rng.uniform(-np.pi, np.pi, size=n)
    mean = true_mean(x, z)

    sigma = sigma_for_x(regime, x)
    y = mean + sigma * rng.normal(size=n)
    return x, z, y, mean, sigma

In [ ]:
# Presentation-ready structural and fold-fit figure.
REGIME_ORDER = ("null", "x_scale")
REGIME_ROW_LABELS = {
    "null": r"null",
    "x_scale": r"alternative",
}
REGIME_DETAIL_LABELS = {
    "null": "null " + sigma_label_for_regime("null"),
    "x_scale": "alternative " + sigma_label_for_regime("x_scale"),
}
X_COLORS = {0: "#2563eb", 1: "#dc2626"}
X_MARKERS = {0: "o", 1: "s"}
FIT_COLOR = "#7c3aed"
TRUTH_COLOR = "#111827"
BAND_COLOR = "#d1d5db"


def sigma_grid_for(regime, x_value, z_grid):
    return sigma_for_x(regime, np.full_like(z_grid, x_value, dtype=int))


def fold_mean_fit_curves(x, z, y, z_grid, *, seed):
    design = trig_design(x, z, MAX_FREQ)
    folds = make_folds(len(y), N_FOLDS, np.random.default_rng(seed))
    curves = {0: [], 1: []}
    fitted_oos = np.zeros(len(y))
    selected_alphas = []
    for holdout in folds:
        train = np.setdiff1d(np.arange(len(y)), holdout)
        beta_m, alpha_m, _ = cv_ridge_fit(design[train], y[train])
        selected_alphas.append(alpha_m)
        fitted_oos[holdout] = design[holdout] @ beta_m
        for x_value in [0, 1]:
            curves[x_value].append(trig_design(np.full_like(z_grid, x_value), z_grid, MAX_FREQ) @ beta_m)
    return curves, fitted_oos, np.array(selected_alphas)


def draw_structural_panel(ax, *, regime, seed, z_grid):
    x, z, y, mean, sigma = simulate(N, regime=regime, seed=seed)
    m0 = true_mean(np.zeros_like(z_grid), z_grid)
    m1 = true_mean(np.ones_like(z_grid), z_grid)

    ax.plot(z_grid, m0, color=X_COLORS[0], linewidth=1.8, label=r"$m(0,z)$")
    ax.plot(z_grid, m1, color=X_COLORS[1], linewidth=1.8, label=r"$m(1,z)$")
    for x_value in [0, 1]:
        mask = x == x_value
        ax.scatter(
            z[mask],
            mean[mask],
            s=13,
            marker=X_MARKERS[x_value],
            color=X_COLORS[x_value],
            alpha=0.62,
            linewidths=0.0,
            label=None,
            zorder=3,
        )
        ax.scatter(
            z[mask],
            y[mask],
            s=12,
            marker=X_MARKERS[x_value],
            facecolors="white" if x_value == 0 else X_COLORS[x_value],
            edgecolors=X_COLORS[x_value],
            linewidths=0.65,
            alpha=0.46,
            label=None,
            zorder=4,
        )
    for zi, mi, yi in zip(z, mean, y):
        ax.plot([zi, zi], [mi, yi], color="0.55", alpha=0.18, linewidth=0.55, zorder=1)

    ax.set_title(r"data and $m(x,t)$", loc="left", pad=3)
    ax.set_ylabel(REGIME_ROW_LABELS[regime])
    ax.set_xlim(z_grid.min(), z_grid.max())
    return x, z, y


def draw_fold_fit_panel(ax, *, x, z, y, regime, x_value, z_grid, fold_seed):
    curves, _, _ = fold_mean_fit_curves(x, z, y, z_grid, seed=fold_seed)
    x_grid = np.full_like(z_grid, x_value)
    truth = true_mean(x_grid, z_grid)
    sigma = sigma_grid_for(regime, x_value, z_grid)

    ax.fill_between(z_grid, truth - 2 * sigma, truth + 2 * sigma, color=BAND_COLOR, alpha=0.45, linewidth=0.0)
    ax.plot(z_grid, truth + 2 * sigma, color="#dc2626", linewidth=0.9, linestyle="--", alpha=0.86, label=r"$m(x,t)\pm2\sigma(x)$")
    ax.plot(z_grid, truth - 2 * sigma, color="#dc2626", linewidth=0.9, linestyle="--", alpha=0.86)
    for fold_idx, curve in enumerate(curves[x_value]):
        ax.plot(z_grid, curve, color=FIT_COLOR, linewidth=1.0, alpha=0.58, label="fold fits" if fold_idx == 0 else None)
    ax.plot(z_grid, truth, color=TRUTH_COLOR, linewidth=1.7, label=r"$m(x,t)$")
    sigma_value = sigma_values_for_regime(regime)[x_value]
    ax.set_title(rf"fold fits, $X={x_value}$, $\sigma({x_value})={sigma_value:g}$", loc="left", pad=3)
    ax.set_xlim(z_grid.min(), z_grid.max())


def save_current_figure(fig, stem):
    if not SAVE_FIGURES:
        return []
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    paths = []
    for fmt in SAVE_FORMATS:
        out = FIGURE_DIR / f"{stem}.{fmt}"
        fig.savefig(out, bbox_inches="tight")
        paths.append(out)
    return paths


def plot_structural_fold_fit_figure():
    z_grid = np.linspace(-np.pi, np.pi, 500)
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(8.7, 5.05),
        sharex=True,
        constrained_layout=False,
    )
    fig.subplots_adjust(left=0.085, right=0.99, bottom=0.05, top=0.85, wspace=0.1, hspace=0.15)
    for row_idx, regime in enumerate(REGIME_ORDER):
        x, z, y = draw_structural_panel(axes[row_idx, 0], regime=regime, seed=7 + 100 * row_idx, z_grid=z_grid)
        for col_idx, x_value in enumerate([0, 1], start=1):
            draw_fold_fit_panel(
                axes[row_idx, col_idx],
                x=x,
                z=z,
                y=y,
                regime=regime,
                x_value=x_value,
                z_grid=z_grid,
                fold_seed=70_007 + 100 * row_idx,
            )
            axes[row_idx, col_idx].set_ylabel("")
    for ax in axes[-1, :]:
        ax.set_xlabel(r"$T$")
    y_min = min(ax.get_ylim()[0] for ax in axes.flat)
    y_max = max(ax.get_ylim()[1] for ax in axes.flat)
    y_pad = 0.02 * (y_max - y_min)
    for ax in axes.flat:
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
        ax.grid(True, axis="y", color="0.90", linewidth=0.8)
        ax.grid(False, axis="x")
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    legend_handles = [
        Line2D([0], [0], marker=X_MARKERS[0], color="none", markerfacecolor="white", markeredgecolor=X_COLORS[0], markeredgewidth=1.15, markersize=6.3, label=r"$Y, X=0$"),
        Line2D([0], [0], marker=X_MARKERS[1], color="none", markerfacecolor=X_COLORS[1], markeredgecolor=X_COLORS[1], markersize=6.3, alpha=0.85, label=r"$Y, X=1$"),
        Line2D([0], [0], color="#dc2626", linewidth=0.9, linestyle="--", alpha=0.86, label=r"$m(x,t)\pm2\sigma(x)$"),
        Line2D([0], [0], color=FIT_COLOR, linewidth=1.2, alpha=0.75, label=r"fold fits of $\widehat m(x,t)$"),
        Line2D([0], [0], color=TRUTH_COLOR, linewidth=1.7, label=r"$m(x,t)$"),
    ]
    fig.legend(legend_handles, [h.get_label() for h in legend_handles], loc="upper center", bbox_to_anchor=(0.52, 0.955), ncol=5, handlelength=1.55, columnspacing=1.0)
    fig.suptitle(r"Cross-fitted mean fits with a binary covariate", y=0.985, fontsize=11.0, fontweight="semibold")
    return fig


structural_fit_fig = plot_structural_fold_fit_figure()
structural_fit_paths = save_current_figure(structural_fit_fig, "structural_fold_fits_with_covariate")
print("Saved:", *(str(path) for path in structural_fit_paths), sep="\n  ")
plt.show()


In [ ]:
def estimate_hsic_u(x, z, y, mean_true, sigma, seed=0):
    """Plug-in and debiased cross-fit U-statistic HSIC estimates."""
    rng = np.random.default_rng(seed)
    n = len(y)
    x_feat = one_hot_binary(x)
    design = trig_design(x, z, MAX_FREQ)
    folds = make_folds(n, N_FOLDS, rng)
    feature_dim = residual_feature_map(np.array([0.0])).shape[1]

    A = np.zeros((n, 2))
    B_naive = np.zeros((n, feature_dim))
    B_debiased = np.zeros((n, feature_dim))
    mean_fit_mse = []
    selected_alphas = []
    selected_v_alphas = []

    for holdout in folds:
        train = np.setdiff1d(np.arange(n), holdout)
        beta_m, alpha_m, _ = cv_ridge_fit(design[train], y[train])
        residual_train = y[train] - design[train] @ beta_m
        residual_hold = y[holdout] - design[holdout] @ beta_m

        score_train = residual_score_map(residual_train)
        beta_v, alpha_v, _ = cv_ridge_fit(design[train], score_train)
        v_hat_hold = design[holdout] @ beta_v
        selected_v_alphas.append(alpha_v)

        mu_x = x_feat[train].mean(axis=0)
        mu_r = residual_feature_map(residual_train).mean(axis=0)

        A[holdout] = x_feat[holdout] - mu_x
        B_naive[holdout] = residual_feature_map(residual_hold) - mu_r
        B_debiased[holdout] = (
            residual_feature_map(residual_hold)
            - mu_r
            - residual_hold[:, None] * v_hat_hold
        )

        fitted_hold = design[holdout] @ beta_m
        selected_alphas.append(alpha_m)
        mean_fit_mse.append(np.mean((fitted_hold - mean_true[holdout]) ** 2))

    naive_H_folds = score_gram_blocks(A, B_naive, folds)
    debiased_H_folds = score_gram_blocks(A, B_debiased, folds)
    naive_score_norm_diagnostic, psi_naive = psi_from_contributions(A, B_naive)
    debiased_score_norm_diagnostic, psi_debiased = psi_from_contributions(A, B_debiased)

    return {
        "naive_hsic_u": crossfit_u_from_blocks(naive_H_folds),
        "debiased_hsic_u": crossfit_u_from_blocks(debiased_H_folds),
        "naive_H_folds": naive_H_folds,
        "debiased_H_folds": debiased_H_folds,
        "naive_score_norm_diagnostic": naive_score_norm_diagnostic,
        "debiased_score_norm_diagnostic": debiased_score_norm_diagnostic,
        "mean_fit_signal_mse": float(np.mean(mean_fit_mse)),
        "selected_alpha": float(np.mean(selected_alphas)),
        "selected_v_alpha": float(np.mean(selected_v_alphas)),
        "psi_naive": psi_naive,
        "psi_debiased": psi_debiased,
        "residual": y - mean_true,
        "x": x,
        "z": z,
    }


In [ ]:
regime_names = {
    "null": "honest null",
    "x_scale": "honest X-scale alternative",
}

if LOAD_SAVED_RESULTS:
    rows = pd.read_csv(RESULT_DIR / "u_statistic_repetitions.csv", keep_default_na=False).to_dict("records")
else:
    rows = []
    for regime in regime_names:
        true_hsic = population_oracle_hsic(regime)
        for rep in range(N_REPS):
            x, z, y, mean_true, sigma = simulate(N, regime=regime, seed=10_000 + rep)
            est = estimate_hsic_u(x, z, y, mean_true, sigma, seed=20_000 + rep)
            rows.append({"regime": regime, "rep": rep, "true_hsic": true_hsic, **{k: est[k] for k in [
                "naive_hsic_u", "debiased_hsic_u", "mean_fit_signal_mse", "selected_alpha", "selected_v_alpha"
            ]}})
    
    pd.DataFrame(rows).to_csv(RESULT_DIR / "u_statistic_repetitions.csv", index=False)

for regime, label in regime_names.items():
    chunk = [r for r in rows if r["regime"] == regime]
    true_hsic = chunk[0]["true_hsic"]
    naive = np.array([r["naive_hsic_u"] for r in chunk])
    deb = np.array([r["debiased_hsic_u"] for r in chunk])
    nerr = np.abs(naive - true_hsic)
    derr = np.abs(deb - true_hsic)
    mmse = np.array([r["mean_fit_signal_mse"] for r in chunk])
    alpha = np.array([r["selected_alpha"] for r in chunk])
    v_alpha = np.array([r["selected_v_alpha"] for r in chunk])
    print(f"{label:28s} true HSIC={true_hsic:.6e}  plug-in_u={naive.mean():.6e}  debiased_u={deb.mean():.6e}")
    print(f"{'':28s} plug-in |err|={nerr.mean():.6e}  debiased |err|={derr.mean():.6e}  deb better={(nerr > derr).mean():.3f}")
    print(f"{'':28s} mean-fit mse={mmse.mean():.4f}  mean alpha={alpha.mean():.4f}  v alpha={v_alpha.mean():.4f}")

ESTIMATOR_SPECS = [
    ("plug-in", "naive_hsic_u", "#dc2626"),
    ("debiased", "debiased_hsic_u", "#2563eb"),
]


def values_by_regime(regime, column):
    return np.array([r[column] for r in rows if r["regime"] == regime], dtype=float)


def truth_by_regime(regime):
    return float(next(r["true_hsic"] for r in rows if r["regime"] == regime))


def style_boxplot(bp, color):
    for body in bp["boxes"]:
        body.set(facecolor=color, edgecolor="#111827", alpha=0.42, linewidth=0.9)
    for key in ["whiskers", "caps"]:
        for line in bp[key]:
            line.set(color="#111827", linewidth=0.9)
    for line in bp["medians"]:
        line.set(color="#111827", linewidth=1.35)
    for flier in bp["fliers"]:
        flier.set(marker="o", markersize=2.5, markerfacecolor=color, markeredgecolor="none", alpha=0.45)


def draw_estimates_vs_oracle(ax):
    regime_labels = [r"null", r"alternative"]
    xpos = np.arange(len(REGIME_ORDER), dtype=float)
    offsets = [-0.16, 0.16]
    for offset, (label, column, color) in zip(offsets, ESTIMATOR_SPECS):
        data = [values_by_regime(regime, column) for regime in REGIME_ORDER]
        bp = ax.boxplot(data, positions=xpos + offset, widths=0.26, patch_artist=True, manage_ticks=False)
        style_boxplot(bp, color)
        ax.scatter([], [], color=color, alpha=0.75, label=label)
    for pos, regime in zip(xpos, REGIME_ORDER):
        ax.hlines(truth_by_regime(regime), pos - 0.39, pos + 0.39, color="#15803d", linewidth=1.7, label="true" if pos == 0 else None)
    ax.axhline(0.0, color="0.25", linewidth=0.9, linestyle=":")
    ax.set_xticks(xpos)
    ax.set_xticklabels(regime_labels)
    ax.set_title(r"$U$ estimates", loc="left", pad=3)
    ax.set_ylabel(r"$\mathrm{HSIC}(X,\epsilon)$")
    ax.legend(loc="upper left", handlelength=1.2)


def draw_error_panel(ax, regime):
    data = []
    labels = []
    colors = []
    truth = truth_by_regime(regime)
    for label, column, color in ESTIMATOR_SPECS:
        data.append(np.abs(values_by_regime(regime, column) - truth))
        labels.append(label)
        colors.append(color)
    positions = np.arange(len(data), dtype=float)
    bp = ax.boxplot(data, positions=positions, widths=0.52, patch_artist=True, manage_ticks=False)
    for box, color in zip(bp["boxes"], colors):
        box.set(facecolor=color, edgecolor="#111827", alpha=0.42, linewidth=0.9)
    for key in ["whiskers", "caps"]:
        for line in bp[key]:
            line.set(color="#111827", linewidth=0.9)
    for line in bp["medians"]:
        line.set(color="#111827", linewidth=1.35)
    for flier, color in zip(bp["fliers"], colors):
        flier.set(marker="o", markersize=2.5, markerfacecolor=color, markeredgecolor="none", alpha=0.45)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_title(REGIME_DETAIL_LABELS[regime], loc="left", pad=3)
    ax.set_ylabel(r"$|\widehat{\mathrm{HSIC}}_U(X,\epsilon)-\mathrm{HSIC}_0(X,\epsilon)|$")


def plot_oracle_comparison_figure():
    fig, axes = plt.subplots(1, 3, figsize=(8.7, 2.85), constrained_layout=False)
    fig.subplots_adjust(left=0.075, right=0.99, bottom=0.20, top=0.82, wspace=0.32)
    draw_estimates_vs_oracle(axes[0])
    draw_error_panel(axes[1], "null")
    draw_error_panel(axes[2], "x_scale")
    for ax in axes.flat:
        ax.grid(True, axis="y", color="0.90", linewidth=0.8)
        ax.grid(False, axis="x")
    fig.suptitle(r"Cross-fit $U$ statistics for $\mathrm{HSIC}(X,\epsilon)$", y=0.97, fontsize=11.0, fontweight="semibold")
    return fig


oracle_comparison_fig = plot_oracle_comparison_figure()
oracle_comparison_paths = save_current_figure(oracle_comparison_fig, "u_statistic_oracle_comparison_with_covariate")
print("Saved:", *(str(path) for path in oracle_comparison_paths), sep="\n  ")
plt.show()


## Delta-Method Coverage Under the Alternative

This final diagnostic repeats the `X`-scale alternative experiment and forms delta-method confidence intervals on the `sqrt(HSIC)` scale. The target is the analytic population oracle for the fixed RFF map, so the coverage calculation is not benchmarked against a noisy sample oracle.

The variance scores use the fold-aware U-statistic convention from `crossfit_u_point_scores` in `multivariate_crossfit_core.py`, and the interval is centered at the cross-fit U statistic.


For each method, set V=||mean(D_i)||², use sqrt(max(U,0)) as the center, and use sqrt(score_var/(n V)) as its standard error. The existing fold-aware U point scores and sample-variance convention are retained. Both coverage and mean width use all 500 repetitions. Invalid V or nonfinite endpoints stop the calculation rather than selecting a subset.

In [ ]:
DELTA_N_REPS = 500
DELTA_REGIME = "x_scale"
DELTA_NOMINAL_LEVELS = np.array([0.50, 0.60, 0.70, 0.80, 0.90, 0.95])

_standard_normal = NormalDist()
DELTA_Z_BY_LEVEL = {
    float(level): _standard_normal.inv_cdf(0.5 + 0.5 * float(level))
    for level in DELTA_NOMINAL_LEVELS
}


def crossfit_u_point_scores(H_folds):
    """NumPy version of the fold-aware U-statistic point scores in the core code."""
    num_folds = len(H_folds)
    scores_by_fold = []
    for k in range(num_folds):
        same_block = H_folds[k][k]
        size_fold = same_block.shape[0]
        same_scores = (np.sum(same_block, axis=1) - np.diag(same_block)) / max(size_fold - 1, 1)

        total = same_scores
        for l in range(num_folds):
            if l != k:
                total = total + np.mean(H_folds[k][l], axis=1)
        scores_by_fold.append(total / num_folds)
    return np.concatenate(scores_by_fold, axis=0)


def delta_sqrt_hsic_ci_from_moments(hsic_u, score_var, n, z_value):
    """Intersect the U-centered squared-norm interval with [0,infinity), then sqrt."""
    if n < 2 or not np.all(np.isfinite([hsic_u, score_var, z_value])):
        raise ValueError("Delta intervals require finite moments and at least two observations.")
    if score_var < 0 or z_value < 0:
        raise ValueError("Score variance and critical value must be nonnegative.")
    se_hsic = 2.0 * np.sqrt(score_var / n)
    lower = max(0.0, hsic_u - z_value * se_hsic)
    upper = hsic_u + z_value * se_hsic
    empty = bool(upper < 0.0)
    sqrt_lower = np.nan if empty else np.sqrt(lower)
    sqrt_upper = np.nan if empty else np.sqrt(upper)
    return {
        "delta_protocol": "u_squared_sqrt",
        "hsic_center": float(hsic_u),
        "sqrt_center": float(np.sqrt(max(hsic_u, 0.0))),
        "clipped_center": bool(hsic_u <= 0.0),  # diagnostic only; interval centers on U
        "score_var": float(score_var),
        "se_hsic": float(se_hsic),
        "hsic_lower": float(lower),
        "hsic_upper": float(upper),
        "sqrt_lower": float(sqrt_lower),
        "sqrt_upper": float(sqrt_upper),
        "sqrt_width": 0.0 if empty else float(sqrt_upper - sqrt_lower),
        "empty": empty,
        "finite": not empty,
    }


def delta_sqrt_hsic_ci_from_blocks(H_folds, z_value):
    """Use fold-aware U point scores for the squared-norm delta interval."""
    hsic_u = crossfit_u_from_blocks(H_folds)
    scores = crossfit_u_point_scores(H_folds)
    if len(scores) < 2 or not np.all(np.isfinite(scores)):
        raise ValueError("Delta intervals require at least two finite point scores.")
    interval = delta_sqrt_hsic_ci_from_moments(
        hsic_u, float(np.var(scores, ddof=1)), len(scores), z_value,
    )
    # Retain V for auditing; it does not normalize this interval.
    interval["hsic_v"] = float(sum(np.mean(block) for row in H_folds for block in row) / len(H_folds)**2)
    return interval


def run_delta_coverage_experiment(n_reps=DELTA_N_REPS, nominal_levels=DELTA_NOMINAL_LEVELS, regime=DELTA_REGIME):
    records = []
    true_hsic = population_oracle_hsic(regime)
    true_sqrt = np.sqrt(max(true_hsic, 0.0))
    for rep in range(n_reps):
        x, z, y, mean_true, sigma = simulate(N, regime=regime, seed=50_000 + rep)
        est = estimate_hsic_u(x, z, y, mean_true, sigma, seed=60_000 + rep)
        for estimator, H_folds in [
            ("plug-in", est["naive_H_folds"]),
            ("debiased", est["debiased_H_folds"]),
        ]:
            for nominal_level in nominal_levels:
                ci = delta_sqrt_hsic_ci_from_blocks(H_folds, z_value=DELTA_Z_BY_LEVEL[float(nominal_level)])
                finite = ci["finite"]
                covered = bool(finite and ci["sqrt_lower"] <= true_sqrt <= ci["sqrt_upper"])
                records.append({
                    "regime": regime,
                    "rep": rep,
                    "estimator": estimator,
                    "nominal_level": float(nominal_level),
                    "true_hsic": true_hsic,
                    "true_sqrt": true_sqrt,
                    "covered": covered,
                    **ci,
                })
    return records


def summarize_delta_records(records):
    """Retain empty intervals as noncoverage and zero width in every denominator."""
    rows = []
    for nominal_level in DELTA_NOMINAL_LEVELS:
        for estimator in ["plug-in", "debiased"]:
            chunk = [r for r in records if r["estimator"] == estimator
                     and np.isclose(r["nominal_level"], nominal_level)]
            widths = np.array([r["sqrt_width"] for r in chunk], dtype=float)
            empty = np.array([r["empty"] for r in chunk], dtype=bool)
            covered = np.array([r["covered"] for r in chunk], dtype=bool)
            if not len(chunk) or not np.all(np.isfinite(widths)) or np.any(widths < 0):
                raise ValueError("Every repetition must have a finite, nonnegative interval width.")
            if np.any(covered[empty]) or np.any(widths[empty] != 0):
                raise ValueError("Empty intervals must have zero width and noncoverage.")
            for r in chunk:
                if not r["empty"] and (not np.all(np.isfinite([r["sqrt_lower"], r["sqrt_upper"]]))
                                       or r["sqrt_lower"] > r["sqrt_upper"]):
                    raise ValueError("A nonempty interval must have ordered finite endpoints.")
            count = len(chunk)
            coverage = float(np.mean(covered))
            rows.append({
                "nominal_level": float(nominal_level), "estimator": estimator,
                "true_sqrt": chunk[0]["true_sqrt"], "n": count,
                "n_finite": int(np.sum(~empty)), "n_empty": int(np.sum(empty)),
                "n_clipped_center": sum(bool(r["clipped_center"]) for r in chunk),
                "finite_rate": float(np.mean(~empty)), "empty_rate": float(np.mean(empty)),
                "invalid_rate": 0.0,
                "coverage_all": coverage,
                "coverage_finite": float(np.mean(covered[~empty])) if np.any(~empty) else np.nan,
                "coverage_se": float(np.sqrt(coverage * (1.0 - coverage) / count)),
                "mean_width": float(np.mean(widths)), "median_width": float(np.median(widths)),
                "width_se": float(np.std(widths, ddof=1) / np.sqrt(count)) if count > 1 else np.nan,
                "mean_center": float(np.mean([r["sqrt_center"] for r in chunk])),
            })
    return rows


In [ ]:
if LOAD_SAVED_RESULTS:
    saved = pd.read_csv(RESULT_DIR / "delta_repetitions.csv")
    if "delta_protocol" not in saved or not saved["delta_protocol"].eq("u_squared_sqrt").all():
        raise ValueError("Saved delta results use an old protocol; set LOAD_SAVED_RESULTS=False to regenerate.")
    delta_records = saved.to_dict("records")
else:
    delta_records = run_delta_coverage_experiment(
        n_reps=DELTA_N_REPS,
        nominal_levels=DELTA_NOMINAL_LEVELS,
        regime=DELTA_REGIME,
    )
    pd.DataFrame(delta_records).to_csv(RESULT_DIR / "delta_repetitions.csv", index=False)

delta_summary = summarize_delta_records(delta_records)
# Backward-compatible display normalization if older records/cells used the old internal label.
for row in delta_summary:
    if row["estimator"] == "naive":
        row["estimator"] = "plug-in"

print(f"Delta-method sqrt-HSIC coverage under {regime_names[DELTA_REGIME]}")
for row in delta_summary:
    print(
        f"level={row['nominal_level']:.2f}  {row['estimator']:8s} "
        f"coverage={row['coverage_all']:.3f} +/- {row['coverage_se']:.3f}  "
        f"empty={row['empty_rate']:.3f}  "
        f"mean width={row['mean_width']:.6e} +/- {row['width_se']:.6e}  "
        f"mean center={row['mean_center']:.6e}"
    )

DELTA_ESTIMATOR_ORDER = ["plug-in", "debiased"]
DELTA_ESTIMATOR_COLORS = {"plug-in": "#dc2626", "debiased": "#2563eb"}
DELTA_ESTIMATOR_MARKERS = {"plug-in": "o", "debiased": "s"}
DELTA_CONFIDENCE_SET_STEM = "delta_method_confidence_sets_with_covariate"


def delta_summary_arrays(estimator):
    chunk = [r for r in delta_summary if r["estimator"] == estimator]
    levels = np.array([r["nominal_level"] for r in chunk])
    coverage = np.array([r["coverage_all"] for r in chunk])
    coverage_se = np.array([r["coverage_se"] for r in chunk])
    widths = np.array([r["mean_width"] for r in chunk])
    width_se = np.array([r["width_se"] for r in chunk])
    return levels, coverage, coverage_se, widths, width_se

pd.DataFrame(delta_summary).to_csv(RESULT_DIR / "delta_summary.csv", index=False)

TABLE_DIR = Path("tables/covariate")
TABLE_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(delta_summary).to_csv(TABLE_DIR / "delta_summary.csv", index=False)


In [ ]:
def plot_delta_confidence_set_horizontal_figure():
    with plt.rc_context({"font.family": "DejaVu Sans"}):
        fig, axes = plt.subplots(1, 2, figsize=(7.1, 2.85), constrained_layout=False)
        fig.subplots_adjust(left=0.095, right=0.985, bottom=0.20, top=0.83, wspace=0.24)
    
        for estimator in DELTA_ESTIMATOR_ORDER:
            levels, coverage, coverage_se, widths, width_se = delta_summary_arrays(estimator)
            color = DELTA_ESTIMATOR_COLORS[estimator]
            marker = DELTA_ESTIMATOR_MARKERS[estimator]
            axes[0].errorbar(
                levels,
                coverage,
                yerr=1.96 * coverage_se,
                marker=marker,
                linewidth=1.45,
                markersize=4.2,
                capsize=2.6,
                capthick=0.9,
                elinewidth=0.9,
                color=color,
                label=estimator,
            )
            axes[1].errorbar(
                levels,
                widths,
                yerr=1.96 * width_se,
                marker=marker,
                linewidth=1.45,
                markersize=4.2,
                capsize=2.6,
                capthick=0.9,
                elinewidth=0.9,
                color=color,
                label=estimator,
            )
    
        axes[0].plot([0.48, 0.97], [0.48, 0.97], color="#111827", linewidth=1.55, linestyle="--", label="nominal", zorder=1)
        axes[0].set_xlim(0.48, 0.97)
        axes[0].set_ylim(0.0, 1.01)
        axes[0].set_xlabel("nominal coverage")
        axes[0].set_ylabel("empirical coverage")
    
        axes[1].set_xlim(0.48, 0.97)
        # Keep the current manuscript axis scale when the interval numbers change.
        axes[1].set_ylim(0.009175104411597702, 0.08183865932976839)
        axes[1].set_xlabel("nominal coverage")
        axes[1].set_ylabel(r"mean $\sqrt{\mathrm{HSIC}(X,\epsilon)}$ CI width")
    
        for ax in axes:
            ax.grid(True, axis="y", color="0.90", linewidth=0.8)
            ax.grid(False, axis="x")
    
        from matplotlib.lines import Line2D
    
        legend_handles = [
            Line2D([0], [0], color="#111827", linewidth=1.55, linestyle="--", label="nominal"),
            Line2D([0], [0], color=DELTA_ESTIMATOR_COLORS["plug-in"], marker=DELTA_ESTIMATOR_MARKERS["plug-in"], linewidth=1.45, markersize=4.8, label="plug-in"),
            Line2D([0], [0], color=DELTA_ESTIMATOR_COLORS["debiased"], marker=DELTA_ESTIMATOR_MARKERS["debiased"], linewidth=1.45, markersize=4.8, label="debiased"),
        ]
        fig.legend(
            legend_handles,
            [handle.get_label() for handle in legend_handles],
            loc="upper center",
            bbox_to_anchor=(0.53, 0.915),
            ncol=3,
            handlelength=1.45,
            columnspacing=1.0,
        )
        return fig

delta_confidence_set_horizontal_fig = plot_delta_confidence_set_horizontal_figure()
delta_confidence_set_horizontal_paths = save_current_figure(
    delta_confidence_set_horizontal_fig,
    f"{DELTA_CONFIDENCE_SET_STEM}_horizontal",
)
print("Saved horizontal:")
for path in delta_confidence_set_horizontal_paths:
    print(f"  {path}")
plt.show()